# Transformers from the Ground Up

## Building Intuition One Step at a Time

This notebook builds the **complete mental model for the Transformer architecture** from first principles — starting with a tiny 3-dimensional semantic space that you can visualise, rotate, and reason about concretely.

Every concept is demonstrated on the same running example:

> **"the cat sat on the mat"**

| Step | Concept                        | Key Idea                                         |
| ---- | ------------------------------ | ------------------------------------------------ |
| 1    | Vocabulary + 3D Embeddings     | Words as points in semantic space                |
| 2    | The Ordering Problem           | Why bags of words lose meaning                   |
| 3    | Sinusoidal Positional Encoding | Adding position with sine waves                  |
| 4    | RoPE                           | Rotating Q/K vectors to encode relative position |
| 5    | Q, K, V + Attention            | Soft dictionary lookup over the sequence         |
| 6    | Multi-Head Attention           | Parallel attention heads                         |
| 7    | Feed-Forward + Layer Norm      | Per-token transformation + stabilisation         |
| 8    | Full Transformer Block         | All components assembled                         |
| 9    | RNN Comparison                 | What came before transformers                    |
| 10   | Mini Language Model            | End-to-end training from scratch                 |
| 11   | Token Generation               | Inference step by step                           |
| 12   | GPT-2 Internals                | A real model, cracked open                       |


In [1]:
# Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("tensorflow", "tensorflow"),
    ("seaborn", "seaborn"),
    ("plotly", "plotly"),
    ("transformers", "transformers"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")
print("Dependencies ready.")

  ok  numpy
  ok  matplotlib


VersionError: Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow/core/framework/attr_value.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings("ignore")

try:
    import plotly.graph_objects as go

    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "font.size": 11})
sns.set_theme(style="whitegrid", palette="muted")
tf.random.set_seed(42)
np.random.seed(42)

print(f"tensorflow {tf.__version__} | plotly available: {HAS_PLOTLY}")

---

## Part 1 — Our Mini Universe: Vocabulary & Embeddings

Before anything else, we need a way to represent words as numbers. This is the job of an **embedding**.

In a real model (e.g. GPT-2), each word lives in a 768-dimensional space — far too many to visualise. Instead we build a **3-dimensional** semantic space where each axis captures a meaningful abstract property:

| Axis      | Meaning      | Low (0)                         | High (1)                        |
| --------- | ------------ | ------------------------------- | ------------------------------- |
| **Dim 0** | Concreteness | Abstract concepts ("the", "on") | Physical objects ("cat", "mat") |
| **Dim 1** | Animacy      | Inanimate ("mat", "fence")      | Living creatures ("cat", "dog") |
| **Dim 2** | Dynamism     | Static states ("mat")           | Active motion ("ran", "jumped") |

Words cluster into natural neighbourhoods: animals share high concreteness + high animacy; verbs cluster in the high-dynamism corner; function words (articles, prepositions) huddle near the origin.


In [ ]:
# ── Vocabulary ─────────────────────────────────────────────────────────────────
VOCAB = {
    "<PAD>": 0,
    "<BOS>": 1,
    "<EOS>": 2,
    "the": 3,
    "a": 4,
    "cat": 5,
    "dog": 6,
    "mat": 7,
    "fence": 8,
    "sat": 9,
    "ran": 10,
    "jumped": 11,
    "on": 12,
    "over": 13,
    "big": 14,
}
IDX2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

# ── 3D Semantic Embeddings ─────────────────────────────────────────────────────
# Each word is a point in [Concreteness, Animacy, Dynamism] space
E = {
    "<PAD>": [0.00, 0.00, 0.00],
    "<BOS>": [0.08, 0.08, 0.15],
    "<EOS>": [0.08, 0.08, 0.15],
    "the": [0.05, 0.04, 0.08],
    "a": [0.05, 0.04, 0.08],
    "cat": [0.91, 0.94, 0.38],
    "dog": [0.88, 0.92, 0.55],
    "mat": [0.96, 0.04, 0.04],
    "fence": [0.93, 0.03, 0.03],
    "sat": [0.34, 0.18, 0.78],
    "ran": [0.28, 0.12, 0.96],
    "jumped": [0.30, 0.14, 0.98],
    "on": [0.14, 0.04, 0.18],
    "over": [0.17, 0.04, 0.24],
    "big": [0.44, 0.04, 0.09],
}

embedding_matrix = tf.constant(
    [E[IDX2WORD[i]] for i in range(VOCAB_SIZE)], dtype=tf.float32
)

# ── Running sentence ───────────────────────────────────────────────────────────
SENTENCE = "the cat sat on the mat"
TOKENS = SENTENCE.split()
TOKEN_IDS = [VOCAB[w] for w in TOKENS]
SEQ_LEN = len(TOKENS)

print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Embedding shape  : {tuple(embedding_matrix.shape)}  (vocab x 3D)")
print(f"Running sentence : {SENTENCE!r}")
print(f"Token IDs        : {TOKEN_IDS}")
print()
print("Embedding matrix — Concreteness, Animacy, Dynamism:")
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")

In [ ]:
# ── Interactive 3D Vocabulary Visualisation ────────────────────────────────────
# Plotly = interactive (drag to rotate). Matplotlib = static fallback.

CATEGORIES = {
    "Article": (["the", "a"], "#636EFA"),
    "Animate Noun": (["cat", "dog"], "#00CC96"),
    "Inanimate Noun": (["mat", "fence"], "#AB63FA"),
    "Verb": (["sat", "ran", "jumped"], "#EF553B"),
    "Preposition": (["on", "over"], "#FFA15A"),
    "Adjective": (["big"], "#19D3F3"),
}

if HAS_PLOTLY:
    fig = go.Figure()
    for cat, (words, color) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        fig.add_trace(
            go.Scatter3d(
                x=xs,
                y=ys,
                z=zs,
                mode="markers+text",
                text=words,
                textposition="top center",
                name=cat,
                marker=dict(
                    size=10,
                    color=color,
                    opacity=0.85,
                    line=dict(color="white", width=1),
                ),
            )
        )
    # Gold path connecting our sentence tokens
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    fig.add_trace(
        go.Scatter3d(
            x=sx,
            y=sy,
            z=sz,
            mode="lines",
            name="Sentence path",
            line=dict(color="gold", width=4, dash="dot"),
        )
    )
    for i, w in enumerate(TOKENS):
        fig.add_trace(
            go.Scatter3d(
                x=[E[w][0]],
                y=[E[w][1]],
                z=[E[w][2]],
                mode="text",
                text=[f"[{i}]"],
                showlegend=False,
                textfont=dict(color="gold", size=11),
            )
        )
    fig.update_layout(
        title=dict(text="<b>3D Semantic Embedding Space</b> — drag to rotate", x=0.5),
        scene=dict(
            xaxis_title="Concreteness",
            yaxis_title="Animacy",
            zaxis_title="Dynamism",
        ),
        width=820,
        height=560,
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    )
    fig.show()
else:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    cmap = {
        "Article": "royalblue",
        "Animate Noun": "mediumseagreen",
        "Inanimate Noun": "mediumpurple",
        "Verb": "tomato",
        "Preposition": "darkorange",
        "Adjective": "deepskyblue",
    }
    for cat, (words, _) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        ax.scatter(xs, ys, zs, s=90, label=cat, color=cmap[cat], alpha=0.9)
        for w in words:
            ax.text(E[w][0], E[w][1], E[w][2], f" {w}", fontsize=9)
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    ax.plot(sx, sy, sz, "o--", color="gold", lw=2, label="Sentence path")
    ax.set_xlabel("Concreteness")
    ax.set_ylabel("Animacy")
    ax.set_zlabel("Dynamism")
    ax.set_title("3D Semantic Embedding Space")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print("Tip: pip install plotly for an interactive, rotatable version")

### Tokenisation

A **tokeniser** converts a raw string into integer IDs the model can process. In our toy system, one word = one token. Production models (GPT-2, LLaMA) use **sub-word tokenisation** (BPE / SentencePiece) that splits rare words into pieces — "transformers" → "transform" + "##ers" — giving a fixed vocabulary that covers any text.

Key operations: `encode(text)` → `list[int]`, `decode(ids)` → `str`, embedding lookup.


In [ ]:
def encode(text: str, add_bos: bool = False, add_eos: bool = False):
    ids = [VOCAB.get(w, VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos:
        ids = [VOCAB["<BOS>"]] + ids
    if add_eos:
        ids = ids + [VOCAB["<EOS>"]]
    return ids


def decode(ids):
    return " ".join(IDX2WORD.get(i, "<??>") for i in ids)


# Demo
phrase = "the big cat jumped over the fence"
enc = encode(phrase)
dec = decode(enc)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {dec!r}")
print()

# 3D embedding vectors for the running sentence
ids = tf.constant(TOKEN_IDS)
embs = tf.gather(embedding_matrix, ids)  # (6, 3)

print(f"Running sentence: {SENTENCE!r}")
print()
print("Embedding vectors  [Concrete, Animate, Dynamic]:")
for i, (w, v) in enumerate(zip(TOKENS, embs.numpy())):
    print(f"  [{i}] {w:<8}  [{v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f}]")

---

## Attention: First Contact

Before positions, before $Q/K/V$ projections, before multi-head — the beating heart of the transformer is one simple idea:

> **Every token looks at every other token and builds a weighted average of them, where the weights answer "how much do I care about you?"**

That's the whole mechanism. Everything else you'll meet later — $Q/K/V$, RoPE, multi-head — is a refinement of _how those weights are computed_. So before we complicate it, let's watch the **raw** version on our sentence, using the embeddings themselves as query, key **and** value — no position, no learned matrices.

Four steps, and the animation **freezes on each one once it's complete** so you can read the state before the next begins:

1. **Pick a query token** — "cat" asks _"who matters to me?"_
2. **Score** it against every token by dot product (similar meaning → high score)
3. **Softmax** the scores into weights that sum to 1
4. **Output** = the weighted sum of the value vectors

Watch where `"cat"` sends its attention.


In [ ]:
# ── Attention, step by step — freezing at each completed step ────────────────
# Minimal attention: query = key = value = the raw embedding (no position, no
# learned projections). We watch the 4 core steps and HOLD on each one once it
# is complete, so you can read the state before the next step begins.

emb_min = tf.gather(embedding_matrix, tf.constant(TOKEN_IDS)).numpy()  # (S, 3)
S_min = len(TOKENS)
QUERY = "cat"
qi = TOKENS.index(QUERY)

scores_min = emb_min @ emb_min[qi]  # dot product vs every token
w_min = np.exp(scores_min - scores_min.max())
w_min /= w_min.sum()
output_min = (w_min[:, None] * emb_min).sum(0)

key_x = np.arange(S_min)
q_x = (S_min - 1) / 2.0
dim_names = ["Concrete", "Animate", "Dynamic"]

PH, REVEAL, HOLD = 4, 18, 12
plen = REVEAL + HOLD
TOTAL = PH * plen + 18


def _phase(f):
    if f >= PH * plen:
        return PH - 1, 1.0, True
    p = f // plen
    loc = f % plen
    return p, min(loc / REVEAL, 1.0), loc >= REVEAL


fig = plt.figure(figsize=(12, 7))
gs = plt.GridSpec(2, 2, height_ratios=[1.5, 1], hspace=0.5, wspace=0.25)
ax_g = fig.add_subplot(gs[0, :])
ax_w = fig.add_subplot(gs[1, 0])
ax_o = fig.add_subplot(gs[1, 1])

captions = [
    'Step 1 — pick a query token: "cat" asks "who matters to me?"',
    'Step 2 — score "cat" against every token by dot product',
    "Step 3 — softmax turns scores into weights that sum to 1",
    "Step 4 — output = weighted sum of the value vectors",
]
done_caps = [
    "Step 1 ✓ query selected",
    "Step 2 ✓ every token scored",
    "Step 3 ✓ weights sum to 1",
    'Step 4 ✓ context-aware vector for "cat" is ready',
]


def update(f):
    p, t, done = _phase(f)
    ax_g.clear()
    ax_w.clear()
    ax_o.clear()

    # ---- top: the attention graph ----
    ax_g.set_xlim(-1, S_min)
    ax_g.set_ylim(-0.6, 1.7)
    ax_g.axis("off")
    if p >= 1:
        if p == 1:
            base = scores_min - scores_min.min()
            base = base / base.max()
            wshow, reveal = base, t
        else:
            wshow, reveal = w_min / w_min.max(), 1.0
        for j in range(S_min):
            lw = 0.5 + 6 * wshow[j] * reveal
            a = min(0.15 + 0.85 * wshow[j] * reveal, 1.0)
            ax_g.plot(
                [q_x, key_x[j]], [0, 1], color="#4c72b0", lw=lw, alpha=a, zorder=1
            )

    for j, tok in enumerate(TOKENS):
        ax_g.scatter(key_x[j], 1, s=520, color="#dddddd", edgecolor="#888", zorder=3)
        ax_g.text(key_x[j], 1, tok, ha="center", va="center", fontsize=9, zorder=4)
        if p >= 2:
            ax_g.text(
                key_x[j],
                1.32,
                f"{w_min[j]:.2f}",
                ha="center",
                fontsize=9,
                color="#c44e52",
                fontweight="bold",
            )

    qsize = 300 + 420 * (t if p == 0 else 1.0)
    ax_g.scatter(q_x, 0, s=qsize, color="gold", edgecolor="#b8860b", zorder=5)
    ax_g.text(
        q_x,
        0,
        QUERY,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        zorder=6,
    )
    ax_g.text(q_x, -0.42, "query", ha="center", fontsize=9, color="#b8860b")
    ax_g.text(
        (S_min - 1) / 2,
        1.6,
        "keys / values (every token)",
        ha="center",
        fontsize=9,
        color="#555",
    )

    # ---- bottom-left: scores -> weights ----
    ax_w.set_xlim(-0.6, S_min - 0.4)
    ax_w.set_ylim(0, 1.05)
    ax_w.set_xticks(key_x)
    ax_w.set_xticklabels(TOKENS, fontsize=8, rotation=20)
    if p == 0:
        ax_w.set_title("scores appear in step 2", fontsize=9, color="#999")
    elif p == 1:
        sc = scores_min - scores_min.min()
        sc = sc / sc.max()
        ax_w.bar(key_x, sc * t, color="#4c72b0", alpha=0.85)
        ax_w.set_title("Step 2 — raw dot-product scores", fontsize=10)
        ax_w.set_ylabel("score (scaled)")
    else:
        sc = scores_min - scores_min.min()
        sc = sc / sc.max()
        blend = (1 - t) * sc + t * w_min if p == 2 else w_min
        colors = ["gold" if j == w_min.argmax() else "#4c72b0" for j in range(S_min)]
        ax_w.bar(key_x, blend, color=colors, alpha=0.85)
        ax_w.set_title(
            (
                "Step 3 — softmax → weights (sum = 1)"
                if p == 2
                else "Step 3 ✓ attention weights"
            ),
            fontsize=10,
        )
        ax_w.set_ylabel("weight")

    # ---- bottom-right: output vector, built in step 4 ----
    ax_o.set_ylim(0, 1.05)
    ax_o.set_xticks(range(3))
    ax_o.set_xticklabels(dim_names, fontsize=8)
    if p < 3:
        ax_o.bar(range(3), [0, 0, 0], color="#55a868")
        ax_o.set_title("output builds in step 4", fontsize=9, color="#999")
    else:
        frac = t * S_min
        k = int(frac)
        part = frac - k
        out = np.zeros(3)
        for j in range(min(k, S_min)):
            out += w_min[j] * emb_min[j]
        if k < S_min:
            out += part * w_min[k] * emb_min[k]
        ax_o.bar(range(3), out, color="#55a868")
        cur = TOKENS[min(k, S_min - 1)]
        ax_o.set_title(
            (
                f'Step 4 — Σ wⱼ·valueⱼ   (adding "{cur}")'
                if not done
                else "Step 4 ✓ context vector"
            ),
            fontsize=10,
        )
        if not done and k < S_min:
            ax_g.scatter(
                key_x[k],
                1,
                s=690,
                facecolor="none",
                edgecolor="#55a868",
                lw=2.5,
                zorder=6,
            )

    fig.suptitle(
        done_caps[p] if done else captions[p],
        fontsize=12,
        fontweight="bold",
        color="#333",
    )


ani = FuncAnimation(
    fig, update, frames=TOTAL, interval=45, blit=False, repeat=True, repeat_delay=1200
)
plt.close(fig)
print(
    "Minimal attention — query = key = value = embedding, no positions, no projections."
)
top3 = w_min.argsort()[::-1][:3]
print(
    '"cat" attends most to: ' + ", ".join(f"{TOKENS[j]} ({w_min[j]:.0%})" for j in top3)
)
HTML(ani.to_jshtml(default_mode="loop"))

#### What just happened — and what's missing

`"cat"` pulled most strongly toward **itself** and **`"dog"`** — its semantic neighbours. Attention found _meaning_ without anyone hand-coding grammar: similar vectors → high dot product → high weight.

But look closely at what we **never used**: _position_. Query, key and value were the raw embeddings. Shuffle the sentence to `"mat the on sat the cat"` and `"cat"` keeps the exact same neighbours and produces the exact same output. **Attention, on its own, is order-blind.**

That is the crack the next section pries open — and the reason every transformer needs a way to inject position.


---

## Part 2 — The Ordering Problem

What happens if we just **sum or average** the token vectors into one sentence representation?

> "the cat sat on the mat"  
> "mat the on sat the cat" ← shuffled nonsense

Both sentences contain exactly the same words. Their mean-pooled embedding is **identical** — the model cannot tell them apart. Position information is not optional; it is load-bearing.


In [ ]:
def bag_of_words(sentence: str) -> tf.Tensor:
    """Mean-pool embeddings — loses all positional information."""
    ids = tf.constant(encode(sentence))
    return tf.reduce_mean(tf.gather(embedding_matrix, ids), axis=0)


sentences = [
    "the cat sat on the mat",
    "mat the on sat the cat",
    "sat cat mat on the the",
]
print("Mean-pooled vectors (all contain the same words):")
for s in sentences:
    v = bag_of_words(s).numpy()
    print(f"  {s!r:<42}  [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}]")

all_same = all(
    np.allclose(bag_of_words(sentences[0]).numpy(), bag_of_words(s).numpy())
    for s in sentences[1:]
)
print(f"\nAll three vectors identical: {all_same}")
print()
print("  -> A model with no positional encoding treats meaningful sentences and")
print("     complete nonsense as the SAME input.  We need positional encoding.")

---

## Part 3 — Positional Encoding

### 3a. Sinusoidal PE (original Transformer, "Attention Is All You Need")

The original paper adds a deterministic signal to each token embedding. For token position $m$ in a model of width $d$:

$$PE_{(m,\, 2i)} = \sin\!\left(\frac{m}{10000^{2i/d}}\right)$$

$$PE_{(m,\, 2i+1)} = \cos\!\left(\frac{m}{10000^{2i/d}}\right)$$

Each dimension pair $(2i, 2i+1)$ oscillates at a different frequency — very high for small $i$ (changes fast along sequence), very low for large $i$ (changes slowly). This lets the model read off rough position from the fast dimensions and fine-grained distance from the slow ones.

**Why it works:** Two positions that are close together have similar PE vectors; distant positions diverge. The dot-product of $PE_m \cdot PE_n$ depends only on the gap $|m - n|$, so attention scores pick up relative distance automatically.


In [ ]:
def sinusoidal_pe(seq_len: int, d_model: int) -> tf.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017)."""
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    positions = np.arange(seq_len)[:, None].astype(np.float32)  # (seq_len, 1)
    dims = np.arange(0, d_model, 2).astype(np.float32)  # 0,2,4,...
    freqs = 1.0 / (10000 ** (dims / d_model))  # decaying frequencies
    pe[:, 0::2] = np.sin(positions * freqs)
    pe[:, 1::2] = np.cos(positions * freqs)
    return tf.constant(pe)


D_VIS = 16  # wider for a nicer heatmap

pe_matrix = sinusoidal_pe(SEQ_LEN, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: heatmap over the 6-token sentence
ax = axes[0]
im = ax.imshow(pe_matrix.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f"d{i}" for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(SEQ_LEN))
ax.set_yticklabels(TOKENS, fontsize=10)
ax.set_title("Sinusoidal PE — our sentence")
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Token position")
plt.colorbar(im, ax=ax)

# Right: show how frequency decays across dimensions
ax2 = axes[1]
pe_long = sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    label = f'dim {i} — {"fast" if i < 4 else "slow"}'
    ax2.plot(pe_long[:, i], label=label, lw=1.8)
ax2.set_title("PE signal per dimension over 50 positions")
ax2.set_xlabel("Token position")
ax2.set_ylabel("PE value")
ax2.legend(fontsize=8)
ax2.set_ylim(-1.1, 1.1)

plt.suptitle("Sinusoidal Positional Encoding", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Show token vectors AFTER adding PE
emb_vectors = tf.gather(embedding_matrix, tf.constant(TOKEN_IDS))  # (6, 3)
# For the viz space (d=3) we use a truncated PE
pe_3d = sinusoidal_pe(SEQ_LEN, 3)
enriched = emb_vectors + pe_3d

print("After adding 3D sinusoidal PE:")
print(f"{'Token':<10}  {'Original':>30}  {'PE':>30}  {'Sum':>30}")
emb_np, pe_np, sum_np = emb_vectors.numpy(), pe_3d.numpy(), enriched.numpy()
for i, w in enumerate(TOKENS):
    ov = emb_np[i].tolist()
    pv = pe_np[i].tolist()
    sv = sum_np[i].tolist()

    def fmt(v):
        return f"[{v[0]:+.3f}, {v[1]:+.3f}, {v[2]:+.3f}]"

    print(f"[{i}] {w:<8}  {fmt(ov):>32}  {fmt(pv):>32}  {fmt(sv):>32}")

#### A nagging question before we move on

Sinusoidal PE looks great in the heatmap above — so why did the field move to RoPE?

The honest answer is a _complaint_: **"you added a beautiful position signal at the input, but by the time attention actually uses it, is it still there?"** Attention doesn't read the input embedding directly. It first sums PE into the token content and then projects the mixture through $W_Q$. If that projection smears the position signal, adding it at the input was partly wasted effort.

Let's not take that on faith — let's measure it, then let that measurement _motivate_ RoPE.


In [ ]:
# ── WHY not just keep sinusoidal PE? Watch the clean signal dilute ────────────
#
# Sinusoidal PE is ADDED at the input.  But attention never uses the input
# directly — it first (a) sums PE with the token content and (b) projects that
# through W_Q.  Does the crisp "closer positions look more similar" signal
# survive?  Let's measure it instead of asserting it.

tf.random.set_seed(0)
d_demo = 16
seq_demo = 12

pe_demo = sinusoidal_pe(seq_demo, d_demo)  # pure position signal (S, d)
content = tf.random.normal((seq_demo, d_demo))  # token content (position-agnostic)
W_Q_demo = tf.random.normal((d_demo, d_demo)) * (1 / np.sqrt(d_demo))

x_in = content + pe_demo  # what actually enters the attention block
q_proj = tf.matmul(x_in, W_Q_demo, transpose_b=True)  # Q = (content + PE) · W_Qᵀ


def pos_sim(mat):
    mat = mat.numpy() if hasattr(mat, "numpy") else np.asarray(mat)
    m = mat / (np.linalg.norm(mat, axis=-1, keepdims=True) + 1e-9)
    return m @ m.T


sim_pe = pos_sim(pe_demo)  # clean, distance-structured
sim_q = pos_sim(q_proj)  # after content mixing + projection

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(
    sim_pe,
    ax=axes[0],
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "cosine sim"},
)
axes[0].set_title("PURE sinusoidal PE\nclean diagonal band = distance-aware")
axes[0].set_xlabel("position")
axes[0].set_ylabel("position")
sns.heatmap(
    sim_q,
    ax=axes[1],
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "cosine sim"},
)
axes[1].set_title("AFTER (content + PE) · W_Qᵀ  (RANDOM W_Q)\nstructure can wash out")
axes[1].set_xlabel("position")
axes[1].set_ylabel("position")
plt.suptitle(
    "Sinusoidal PE can dilute once it is summed with content and projected",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()


def monotonicity(sim):
    n = sim.shape[0]
    closeness = np.array([[-abs(i - j) for j in range(n)] for i in range(n)])
    return np.corrcoef(closeness.flatten(), sim.flatten())[0, 1]


print('"Closer positions = more similar" correlation:')
print(
    f"  Pure PE (at the input)        : {monotonicity(sim_pe):+.3f}   <- strong, clean"
)
print(f"  After content + W_Q projection: {monotonicity(sim_q):+.3f}   <- weaker")
print()
print("CAVEAT: W_Q here is RANDOM (untrained). A LEARNED W_Q can preserve much more")
print("of the position signal, so this is an illustration of the RISK, not a proof")
print("that additive PE is broken — trained models with sinusoidal PE work fine.")
print()
print('The principled case for RoPE is NOT "additive PE fails". It is:')
print("  * position is injected AFTER projection, so W_Q cannot dilute it, and")
print("  * the Q.K score depends only on the relative gap (m - n) BY CONSTRUCTION,")
print("    which also tends to extrapolate better to longer sequences than training.")

### 3b. RoPE — Rotary Positional Embeddings

Sinusoidal PE **adds** a fixed vector to the token embedding. RoPE does something more elegant: it **rotates** the Query and Key vectors just before the dot-product, by an angle that depends on absolute position. The rotation cancels in a relative way — only the gap $m - n$ survives.

#### The θᵢ formula

For a model of dimension $d$, pair index $i \in \{0, 1, \ldots, d/2 - 1\}$:

$$\theta_i = \frac{1}{10000^{2i/d}}$$

This is identical to the frequency ladder in sinusoidal PE. Pair $i = 0$ has $\theta_0 = 1.0$ (full rotation per step); pair $i = d/2 - 1$ has a very small $\theta$ (barely moves).

#### Rule 1 — Rotation amount

Token at position $m$ gets its pair $i$ rotated by $m \cdot \theta_i$ radians:

$$\begin{bmatrix} q_{2i}' \\ q_{2i+1}' \end{bmatrix} = \begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{bmatrix} \begin{bmatrix} q_{2i} \\ q_{2i+1} \end{bmatrix}$$

#### Rule 2 — Why it encodes relative distance

The dot product of a rotated Q at position $m$ with a rotated K at position $n$ depends **only on** $m - n$:

$$\langle R_m \mathbf{q},\ R_n \mathbf{k} \rangle = \langle R_{m-n} \mathbf{q},\ \mathbf{k} \rangle$$

This is because rotation matrices satisfy $R_m^T R_n = R_{n-m}$.

#### Production trick — Split-half vectorisation

The rotation above pairs **adjacent** dimensions $(q_{2i}, q_{2i+1})$ — that's the convention in the original paper and in `rope_rotate` below. Most GPU implementations (GPT-NeoX, LLaMA, HF `transformers`) instead avoid building any rotation matrix and use the identity:

$$\text{rotate}(\mathbf{x}) = \mathbf{x} \cdot \cos\theta + \text{rotate\_half}(\mathbf{x}) \cdot \sin\theta$$

where `rotate_half(x) = [-x_{d/2}, ..., -x_{d-1}, x_0, ..., x_{d/2-1}]`. This runs as two element-wise multiply-add operations — extremely GPU-friendly.

> **Important subtlety:** `rotate_half` does **not** pair adjacent dimensions. It pairs dimension $i$ with $i + d/2$ (first half with second half). So it is a _different pairing convention_ from the adjacent-pair matrix above — **not** literally the same formula. The two are nevertheless equivalent **up to a fixed permutation of the dimensions**: relabel the axes and one becomes the other, and both give the identical relative-distance property. The next code cell proves this equivalence numerically rather than asking you to take it on faith.


In [ ]:
# ── RoPE θᵢ values — frequency decay visualisation ───────────────────────────
D_ROPE = 6  # for 3D viz: 3 rotation pairs
half = D_ROPE // 2

thetas = np.array([1.0 / (10000 ** (2 * i / D_ROPE)) for i in range(half)])
print(f"θ values for d={D_ROPE}: {thetas}")
print()

# Show how fast each pair rotates per token step
steps = 50
angles = np.outer(np.arange(steps), thetas)  # (50, 3) — angle for each pos & pair

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
pair_labels = [f"Pair {i}  θ={t:.4f}" for i, t in enumerate(thetas)]
for i in range(half):
    ax.plot(angles[:, i], label=pair_labels[i], lw=2)
ax.set_title("Accumulated rotation angle per pair (radians)")
ax.set_xlabel("Token position m")
ax.set_ylabel("Angle  m·θᵢ  (rad)")
ax.legend(fontsize=9)

ax2 = axes[1]
for i in range(half):
    ax2.plot(np.cos(angles[:, i]), label=f"cos — pair {i}", lw=1.5, ls="-")
    ax2.plot(np.sin(angles[:, i]), label=f"sin — pair {i}", lw=1.5, ls="--", alpha=0.7)
ax2.set_title("cos(m·θᵢ) and sin(m·θᵢ) — the rotation weights")
ax2.set_xlabel("Token position m")
ax2.legend(fontsize=8, ncol=2)
ax2.set_ylim(-1.1, 1.1)

plt.suptitle("RoPE frequency ladder", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print()
print("Pair 0 (fast): completes a full 2π rotation in ~6 steps")
print("Pair 1 (mid) : ~6000 steps for one full rotation")
print("Pair 2 (slow): ~60000 steps for one full rotation")
print()
print("-> Each pair acts like a hand on a clock running at a different speed.")
print('   The model reads position from the composite "clock face".')

### 3d. Building the RoPE animation the way you'd actually discover it

Nobody arrives at a good visualisation in one shot. You build something crude, stare at it, mutter _"…I want more,"_ and add exactly the one thing that was missing. The animation a few cells down is the _final_ form — but if we drop you straight into it, the payoff ("why are the discs **stacked**?") lands flat, because you were never made to _want_ the stacking.

So we'll build it in front of you, refinement by refinement, each step triggered by the complaint the previous step provoked:

| Step                       | What we add                                                              | The complaint that forces the next step                             |
| -------------------------- | ------------------------------------------------------------------------ | ------------------------------------------------------------------- |
| **1. One dial**            | a single pair as a 2D dial, stepping `m`                                 | _"…but a real vector has many pairs at different speeds."_          |
| **2. Stack the pairs**     | one disc per pair, at its own height                                     | _"…but that's just one token — show me a whole sentence."_          |
| **3. One tower per token** | a column per position, activating left→right                             | _"…but I'm watching cos/sin scaffolding, not the actual q values."_ |
| **4. Rotate the real q**   | disc tip **is** the `(q₂ᵢ, q₂ᵢ₊₁)` pair; matrix prints the exact numbers | _(the payoff — nothing left to complain about)_                     |

Run the next cell for **Attempt 1**, then read the complaint it prints.


#### Steps 2, 3 and 4 — all at once, because the complaints compound

Rather than render three more animations, the next cell jumps to the **final** form that answers every complaint above simultaneously:

- **Step 2 (stacking)** — each of the 3 pairs gets its **own disc at its own height**, so fast/medium/slow are visible in one glance. _That_ is why the discs are stacked: one axis for "which pair," separate from the rotation itself.
- **Step 3 (one tower per token)** — a **column per position**, activating **left→right** and locking once placed, mirroring how a sequence is consumed.
- **Step 4 (real data)** — and the fix a sharp reader always demands: **the disc tip is the actual `(q₂ᵢ, q₂ᵢ₊₁)` value**, and the matrix underneath prints those exact numbers as they rotate — _not_ `cos/sin` scaffolding. Because rotation preserves length, each disc's **radius is fixed**; only the **angle** changes with position. That single invariant is the whole reason RoPE is "just a rotation."

These are the same `q` values you'll see change in **Part 4b (RoPE inside attention)** — the animation and that cell are now telling one continuous story.


In [ ]:
# ── Attempt 1 — the crudest possible RoPE picture: one dial ──────────────────
# Plot a SINGLE dimension pair (pair 0) as a 2D dial, stepped across positions.
# It's primitive on purpose — but it makes exactly one thing undeniable.

th0 = 1.0 / (10000 ** (0 / 6))  # θ for pair 0
circle = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), subplot_kw={"aspect": "equal"})
for ax, m in zip(axes, range(4)):
    ang = m * th0
    ax.plot(np.cos(circle), np.sin(circle), color="lightgray", lw=1)
    ax.plot([0, np.cos(ang)], [0, np.sin(ang)], color="#1f77b4", lw=3)
    ax.plot(np.cos(ang), np.sin(ang), "o", color="#1f77b4", ms=10)
    ax.set_title(f"pos m={m}\nangle = {np.degrees(ang):.1f}°", fontsize=10)
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.axhline(0, color="gray", lw=0.4)
    ax.axvline(0, color="gray", lw=0.4)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle(
    "Attempt 1 — one dial, stepping the position. It rotates further as m grows.",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Good — but a real query vector has 3 pairs, and each rotates at a DIFFERENT")
print("speed (fast / medium / slow). One dial hides that.")
print("I want more: stack all three pairs so I can compare their speeds at a glance.")

In [ ]:
# ── Animated RoPE — watch each pair rotate as position increases ───────────────
# We visualise 3 rotation pairs as stacked discs at different heights.
# Heights are spread generously so the degree labels never overlap.

RADII = [1.0, 0.7, 0.4]  # slightly reduced so discs stay within frame
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]
HEIGHTS = [1.2, 0.6, 0.0]  # 0.6 gap between discs (was 0.3) → no overlap
N_FRAMES = 60

fig3d = plt.figure(figsize=(7, 9))  # taller figure for vertical breathing room
ax3d = fig3d.add_subplot(111, projection="3d")

circle_t = np.linspace(0, 2 * np.pi, 200)

# Camera angle: tilt down slightly so all three discs read clearly
ax3d.view_init(elev=22, azim=-60)


def init_rope_anim():
    ax3d.cla()
    ax3d.view_init(elev=22, azim=-60)
    ax3d.set_xlim(-1.5, 1.5)
    ax3d.set_ylim(-1.5, 1.5)
    ax3d.set_zlim(-0.2, 1.6)
    ax3d.set_xlabel("cos component", labelpad=6)
    ax3d.set_ylabel("sin component", labelpad=6)
    ax3d.set_zlabel("Pair index", labelpad=6)
    return []


def update_rope(frame):
    m = frame  # current token position
    ax3d.cla()
    ax3d.view_init(elev=22, azim=-60)
    ax3d.set_xlim(-1.5, 1.5)
    ax3d.set_ylim(-1.5, 1.5)
    ax3d.set_zlim(-0.2, 1.6)
    ax3d.set_xlabel("cos component", labelpad=6)
    ax3d.set_ylabel("sin component", labelpad=6)
    ax3d.set_zlabel("Pair index", labelpad=6)
    ax3d.set_title(f"RoPE rotation  — token position m = {m}", pad=12)

    for i, (r, col, h) in enumerate(zip(RADII, COLORS, HEIGHTS)):
        # Faint guide circle
        ax3d.plot(
            r * np.cos(circle_t),
            r * np.sin(circle_t),
            [h] * 200,
            color=col,
            alpha=0.18,
            lw=1,
        )
        # Rotating hand
        angle = m * thetas[i]
        tip_x = r * np.cos(angle)
        tip_y = r * np.sin(angle)
        ax3d.plot([0, tip_x], [0, tip_y], [h, h], color=col, lw=2.5)
        ax3d.scatter(
            [tip_x],
            [tip_y],
            [h],
            color=col,
            s=60,
            zorder=5,
            label=f"Pair {i}  θ={thetas[i]:.4f}",
        )
        # Label offset: push out along the hand direction + up by 0.10 in z
        ax3d.text(
            tip_x + 0.08 * np.cos(angle),
            tip_y + 0.08 * np.sin(angle),
            h + 0.10,  # was h+0.04; now well above each disc
            f"{np.degrees(angle):.1f}°",
            fontsize=8,
            color=col,
        )

    ax3d.legend(loc="upper left", fontsize=8)
    return []


ani = FuncAnimation(
    fig3d,
    update_rope,
    frames=N_FRAMES,
    init_func=init_rope_anim,
    interval=120,
    blit=False,
)

plt.close(fig3d)  # prevent static display
HTML(ani.to_jshtml(default_mode="loop"))

In [ ]:
# ── RoPE implementation + relative-distance proof ────────────────────────────


def rope_rotate(x, m: int, thetas_arr: np.ndarray) -> np.ndarray:
    """Rotate vector x (shape: d) at token position m using RoPE.
    Uses the paper-style adjacent pairing: (x_0, x_1), (x_2, x_3), ...
    """
    x_rot = np.array(x, dtype=np.float32).copy()
    for i, th in enumerate(thetas_arr):
        angle = m * th
        c, s = math.cos(angle), math.sin(angle)
        a, b = x_rot[2 * i], x_rot[2 * i + 1]
        x_rot[2 * i] = c * a - s * b
        x_rot[2 * i + 1] = s * a + c * b
    return x_rot


def rotate_half(x: tf.Tensor) -> tf.Tensor:
    """Production-style split-half helper (GPU-vectorised).
    Pairs dimension i with i + d/2 — NOT adjacent dims."""
    h = x.shape[-1] // 2
    return tf.concat([-x[..., h:], x[..., :h]], axis=-1)


def rope_apply_production(q: tf.Tensor, cos_val: tf.Tensor, sin_val: tf.Tensor):
    return q * cos_val + rotate_half(q) * sin_val


# ── Relative-distance proof ────────────────────────────────────────────────────
np.random.seed(0)
d_test = 6
half_d = d_test // 2
th = np.array([1.0 / (10000 ** (2 * i / d_test)) for i in range(half_d)])

q = np.random.randn(d_test).astype(np.float32)
k = np.random.randn(d_test).astype(np.float32)

for m, n in [(0, 2), (3, 5), (10, 12), (100, 102)]:
    q_rot = rope_rotate(q, m, th)
    k_rot = rope_rotate(k, n, th)

    # dot(R_m q, R_n k) should equal dot(R_{m-n} q, k)
    lhs = float((q_rot * k_rot).sum())
    rhs = float((rope_rotate(q, m - n, th) * k).sum())
    print(
        f"m={m:3d}  n={n:3d}  gap={m-n:4d} | "
        f"dot(R_m q, R_n k)={lhs:+.6f}  dot(R_{{m-n}} q, k)={rhs:+.6f}  "
        f"equal={abs(lhs-rhs) < 1e-5}"
    )

print()
print("Key insight: the dot product only depends on the GAP (m - n),")
print("not on the absolute positions m and n separately.")

# ── Two conventions, one rotation: adjacent-pair vs split-half ───────────────
# rope_rotate pairs ADJACENT dims (2i, 2i+1); production rotate_half pairs each
# dim i with i + d/2. They are the SAME rotation up to a fixed permutation of
# the dimensions — let's verify that numerically instead of asserting it.
m_demo = 5
ang = m_demo * th  # (half_d,) angle per pair
cos_v = tf.constant(np.concatenate([np.cos(ang)] * 2), dtype=tf.float32)
sin_v = tf.constant(np.concatenate([np.sin(ang)] * 2), dtype=tf.float32)

# Permutation: adjacent order [a0,b0,a1,b1,a2,b2] -> split-half order [a0,a1,a2,b0,b1,b2]
perm = list(range(0, d_test, 2)) + list(range(1, d_test, 2))  # [0,2,4,1,3,5]
inv_perm = np.argsort(perm)

adjacent = rope_rotate(q, m_demo, th)  # (x_2i, x_2i+1) pairing
split = rope_apply_production(tf.constant(q[perm]), cos_v, sin_v).numpy()[
    inv_perm
]  # (i, i+d/2), re-aligned

print()
print("adjacent-pair rotation :", adjacent.round(4))
print("split-half rotation    :", split.round(4))
print("identical after permutation:", np.allclose(adjacent, split, atol=1e-5))
print("-> Production code and the textbook rotation matrix are the SAME operation;")
print("   they differ only in which dimensions are paired, a choice fixed at init.")

---

## Part 4 — Queries, Keys & Values

The transformer's attention mechanism is a **soft dictionary lookup**. Imagine a Python dict where keys aren't exact strings but semantic vectors. You send in a query and the dict returns a _weighted blend_ of all values, with weights determined by how closely the query matches each key.

| Component     | Intuition                | Created by    |
| ------------- | ------------------------ | ------------- |
| **Q** (Query) | "What am I looking for?" | $W_Q \cdot x$ |
| **K** (Key)   | "What do I contain?"     | $W_K \cdot x$ |
| **V** (Value) | "What do I contribute?"  | $W_V \cdot x$ |

Every token simultaneously acts as Q (asking a question), K (advertising its content), and V (providing its payload). The model learns $W_Q$, $W_K$, $W_V$ during training.

### Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ scaling prevents the dot-products from growing so large that softmax saturates into near-one-hot distributions (vanishing gradients).


In [ ]:
# ── Q / K / V projections in 3D space ────────────────────────────────────────
tf.random.set_seed(7)
D_MODEL = 3  # our 3D visualisation space

# Small learned projection matrices (3 → 3 for visualisation)
W_Q = tf.random.normal((D_MODEL, D_MODEL)) * 0.5
W_K = tf.random.normal((D_MODEL, D_MODEL)) * 0.5
W_V = tf.random.normal((D_MODEL, D_MODEL)) * 0.5

embs = tf.gather(embedding_matrix, tf.constant(TOKEN_IDS))  # (6, 3)

Q = tf.matmul(embs, W_Q, transpose_b=True)  # (6, 3)
K = tf.matmul(embs, W_K, transpose_b=True)
V = tf.matmul(embs, W_V, transpose_b=True)

fig = plt.figure(figsize=(15, 4))
titles = ["Input Embeddings", "Queries  (W_Q·x)", "Keys  (W_K·x)", "Values  (W_V·x)"]
arrays = [embs, Q, K, V]
colors = plt.cm.tab10(np.linspace(0, 0.6, SEQ_LEN))

for idx, (title, arr) in enumerate(zip(titles, arrays)):
    ax = fig.add_subplot(1, 4, idx + 1, projection="3d")
    for j, (word, vec) in enumerate(zip(TOKENS, arr.numpy())):
        ax.scatter(*vec, color=colors[j], s=70, zorder=5)
        ax.text(*vec, f" {word}", fontsize=7, color=colors[j])
    ax.set_title(title, fontsize=9, pad=6)
    ax.set_xlabel("d₀", fontsize=7)
    ax.set_ylabel("d₁", fontsize=7)
    ax.set_zlabel("d₂", fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(
    "How W_Q, W_K, W_V rotate/stretch the embedding space", fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
def scaled_attention(Q: tf.Tensor, K: tf.Tensor, V: tf.Tensor, mask=None):
    """Scaled dot-product attention. Returns (output, attn_weights).
    Q, K, V: (seq_len, d_k)
    """
    d_k = Q.shape[-1]

    # Step 1: raw scores  (seq_len, seq_len)
    scores = tf.matmul(Q, K, transpose_b=True) / math.sqrt(d_k)
    print(f"  Raw score matrix (QKᵀ / √d_k):\n  {scores.numpy().round(3)}")

    # Step 2: optional causal mask (decoder)
    if mask is not None:
        scores = tf.where(mask, tf.constant(float("-inf"), scores.dtype), scores)

    # Step 3: softmax → attention weights
    attn_w = tf.nn.softmax(scores, axis=-1)

    # Step 4: weighted sum of Values
    out = tf.matmul(attn_w, V)
    return out, attn_w


print("=== Step-by-step attention on our sentence ===")
print(
    f"Q shape: {tuple(Q.shape)}  K shape: {tuple(K.shape)}  V shape: {tuple(V.shape)}\n"
)

out, attn_w = scaled_attention(Q, K, V)

print(f"\n  Attention weight matrix (row = query token, col = key token):")
print(f"  Tokens: {TOKENS}")
print(f"  {attn_w.numpy().round(3)}")
print(f"\n  Output shape: {tuple(out.shape)}")

In [ ]:
# ── Attention heatmap visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Standard attention (all tokens see all)
ax = axes[0]
w = attn_w.numpy()
sns.heatmap(
    w,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=TOKENS,
    yticklabels=TOKENS,
    linewidths=0.5,
    cbar_kws={"label": "attention weight"},
)
ax.set_title("Bidirectional Attention (encoder-style)")
ax.set_xlabel("Key token")
ax.set_ylabel("Query token")
ax.tick_params(axis="x", rotation=30)

# Causal masked attention (decoder-style — future tokens masked)
causal_mask = tf.constant(np.triu(np.ones((SEQ_LEN, SEQ_LEN)), k=1).astype(bool))
_, causal_w = scaled_attention(Q, K, V, mask=causal_mask)
ax2 = axes[1]
sns.heatmap(
    causal_w.numpy(),
    ax=ax2,
    annot=True,
    fmt=".2f",
    cmap="Oranges",
    xticklabels=TOKENS,
    yticklabels=TOKENS,
    linewidths=0.5,
    cbar_kws={"label": "attention weight"},
)
ax2.set_title("Causal Masked Attention (decoder-style / GPT)")
ax2.set_xlabel("Key token")
ax2.set_ylabel("Query token")
ax2.tick_params(axis="x", rotation=30)

plt.suptitle(
    'Attention weights for: "the cat sat on the mat"', fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

print()
print("Encoder: each token can attend to ALL other tokens.")
print("Decoder: token at position i can ONLY attend to tokens 0..i")
print("         (future tokens are masked to -inf before softmax).")

### 🧪 Your turn — attention

You've watched attention; now drive it. In the next cell, change one variable and **predict the answer before you run it**. Doing beats reading.


In [ ]:
# 🧪 EXERCISE 1 — attention by hand
# 👉 CHANGE `my_query` to any token, and PREDICT its top attention target
#    BEFORE running.  TOKENS = ['the', 'cat', 'sat', 'on', 'the', 'mat']
my_query = "cat"  # ← try 'sat', 'mat', 'on', ...

qi = TOKENS.index(my_query)
scores_ex = tf.matmul(Q, K, transpose_b=True) / math.sqrt(Q.shape[-1])
w_ex = tf.nn.softmax(scores_ex, axis=-1).numpy()[qi]

print(f'"{my_query}" (position {qi}) attends most to:')
for r in w_ex.argsort()[::-1][:3]:
    print(f"   {TOKENS[r]:<5} {w_ex[r]:.0%}")
print(
    "\nDid your prediction match? Attention just answers: which tokens does THIS token pull from?"
)

In [ ]:
# ── RoPE applied to Q and K inside attention — step 1: rotate Q and K ────────
#
# This cell shows the EXACT moment RoPE does its work.
# After W_Q and W_K project the embeddings, RoPE rotates the result by an angle
# that encodes absolute position. The rotated Q and K then go into the
# dot-product — V is never touched.


def apply_rope_to_qk(Q: tf.Tensor, K: tf.Tensor, thetas_arr: np.ndarray) -> tuple:
    """Apply RoPE to Q and K (both shape seq_len × d) using adjacent-pair
    rotation. Robust to an odd d (our 3D viz): the last coordinate is left
    unrotated. Returns (Q_rot, K_rot)."""
    Q = Q.numpy() if hasattr(Q, "numpy") else np.asarray(Q)
    K = K.numpy() if hasattr(K, "numpy") else np.asarray(K)
    seq_len, d = Q.shape
    n_pairs = d // 2
    positions = np.arange(seq_len, dtype=np.float32)  # (S,)

    def rope(x):
        out = x.copy()
        for i in range(n_pairs):
            ang = positions * float(thetas_arr[i])  # (S,)
            cos, sin = np.cos(ang), np.sin(ang)
            a, b = x[:, 2 * i], x[:, 2 * i + 1]
            out[:, 2 * i] = a * cos - b * sin
            out[:, 2 * i + 1] = a * sin + b * cos
        return out  # last dim (odd d) unchanged

    return tf.constant(rope(Q)), tf.constant(rope(K))


# Projected Q and K from the previous cell (3D viz space)
th_vis = np.array([1.0 / (10000 ** (2 * i / D_MODEL)) for i in range(D_MODEL // 2)])
Q_raw, K_raw = Q, K
Q_rot, K_rot = apply_rope_to_qk(Q_raw, K_raw, th_vis)

# Compare raw vs. rotated Q vectors for each token
print(f'{"Token":<8}  {"Q_raw":>26}  {"Q_rotated (RoPE)":>26}  {"Δ norm":>8}')
print("  " + "─" * 74)
q_raw_np, q_rot_np = Q_raw.numpy(), Q_rot.numpy()
for i, tok in enumerate(TOKENS):
    qr, qn = q_raw_np[i], q_rot_np[i]
    delta = np.linalg.norm(qn - qr)
    print(
        f"  {tok:<8}  [{qr[0]:+.3f}, {qr[1]:+.3f}, {qr[2]:+.3f}]  "
        f"[{qn[0]:+.3f}, {qn[1]:+.3f}, {qn[2]:+.3f}]  {delta:>8.4f}"
    )

print()
print(
    'Notice: "the" at position 0 has m=0, so angle=0 → Q_rotated = Q_raw (no change).'
)
print("All other tokens are rotated by m×θᵢ, which bakes position into the scores.")

#### Does that rotation actually change what attends to what?

We've rotated Q and K, but the only thing that matters is whether the **attention weights** move. Same projections, one difference — RoPE applied or not — side by side.


In [ ]:
# ── RoPE in attention — step 2: the payoff on attention weights ──────────────
# Identical Q/K projections; the ONLY difference is whether RoPE was applied.
scores_raw = tf.matmul(Q_raw, K_raw, transpose_b=True) / math.sqrt(D_MODEL)
scores_rot = tf.matmul(Q_rot, K_rot, transpose_b=True) / math.sqrt(D_MODEL)
attn_raw = tf.nn.softmax(scores_raw, axis=-1)
attn_rot = tf.nn.softmax(scores_rot, axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, w, title in zip(
    axes, [attn_raw, attn_rot], ["Attention WITHOUT RoPE", "Attention WITH RoPE"]
):
    sns.heatmap(
        w.numpy(),
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=TOKENS,
        yticklabels=TOKENS,
        linewidths=0.5,
    )
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle(
    "RoPE shifts attention weights by baking position into Q·K scores",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

### 4c. Discovering the attention formula — softmax and √dₖ

We keep writing $\text{softmax}(QK^T / \sqrt{d_k})$ as if it fell from the sky. It didn't — both pieces are _forced_ on us. Before running the next cell, commit to a guess.

#### 🔮 Predict first

1. Raw $QK^T$ scores can be negative and don't sum to 1. **What single operation turns an arbitrary score vector into a probability distribution?**
2. In a real model $d_k$ is 64–128. The dot product of two random vectors grows with dimension. **What happens to `softmax` when the scores get very large — and why would that be catastrophic during training?**

Write your guess down, then run the cell and watch each choice become inevitable.


In [ ]:
# ── DISCOVERING softmax and the √dₖ scale — decision 1: why softmax? ─────────
#
# Two design choices are baked into  softmax(QKᵀ / √dₖ).  Instead of accepting
# them, we rediscover each in its own step. First: why softmax at all?

raw = tf.matmul(Q_rot, K_rot, transpose_b=True)[
    0
].numpy()  # one query row of raw scores
print("Raw QKᵀ scores for one query row:")
print("  ", raw.round(3))
print(f"  sum = {raw.sum():+.3f}  (not 1)  and some are negative → NOT a probability.")
print(
    "  softmax fixes both at once: exp() makes them positive, then normalise to sum = 1."
)

#### Decision 2 — why divide by √dₖ?

Softmax alone isn't enough. The dot product of two random vectors has variance that **grows with dimension** — so at realistic $d_k$ the raw scores get huge. First, watch the variance grow.


In [ ]:
# ── Decision 2, step 1: dot-product variance grows with dimension ────────────
dims = [4, 16, 64, 256, 1024]
print(f'{"dₖ":>6} | {"std(QKᵀ) unscaled":>18} | {"std after /√dₖ":>15}')
print("  " + "─" * 46)
peak_unscaled, peak_scaled = [], []
for d in dims:
    q = tf.random.normal((4000, d))
    k = tf.random.normal((4000, d))
    dot = tf.reduce_sum(q * k, axis=-1).numpy()
    print(f"{d:>6} | {dot.std():>18.2f} | {(dot / math.sqrt(d)).std():>15.2f}")
    # peak softmax prob over an 8-way row, averaged
    qr, kr = tf.random.normal((500, 8, d)), tf.random.normal((500, 8, d))
    s_un = tf.reduce_sum(qr * kr, axis=-1)
    s_sc = s_un / math.sqrt(d)
    peak_unscaled.append(
        float(tf.reduce_mean(tf.reduce_max(tf.nn.softmax(s_un, axis=-1), axis=-1)))
    )
    peak_scaled.append(
        float(tf.reduce_mean(tf.reduce_max(tf.nn.softmax(s_sc, axis=-1), axis=-1)))
    )
print("  Unscaled variance grows like dₖ; dividing by √dₖ pins it ≈ 1 at every width.")

#### The consequence — saturation kills the gradient

Big scores push softmax toward a one-hot spike. A one-hot softmax has almost no slope, so the gradient flowing back through it vanishes and the layer stops learning. Let's measure the gradient with and without the scale.


In [ ]:
# ── Decision 2, step 2: the CONSEQUENCE — saturation kills the gradient ──────
d_big = 256
q = tf.Variable(tf.random.normal((8, d_big)))
k = tf.random.normal((8, d_big))
for label, scale in [("WITHOUT /√dₖ", 1.0), ("WITH /√dₖ", math.sqrt(d_big))]:
    with tf.GradientTape() as tape:
        p = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / scale, axis=-1)
        loss = tf.reduce_sum(tf.reduce_max(p, axis=-1))
    g = tape.gradient(loss, q)
    print(
        f"  {label:<14}: mean peak prob = {float(tf.reduce_mean(tf.reduce_max(p, axis=-1))):.3f}   "
        f"‖∂loss/∂q‖ = {float(tf.norm(g)):.2e}"
    )
print()
print(
    "WITHOUT scaling: softmax ≈ one-hot (peak→1) → gradient ≈ 0 → the layer cannot learn."
)
print("WITH   scaling: distribution stays soft → gradient flows → training works.")

# Visual: softmax peak vs. dimension
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, peak_unscaled, "o-", color="tomato", lw=2, label="unscaled QKᵀ")
ax.plot(dims, peak_scaled, "o-", color="seagreen", lw=2, label="scaled  QKᵀ/√dₖ")
ax.axhline(1 / 8, color="gray", ls="--", lw=1, label="uniform (1/8)")
ax.set_xscale("log", base=2)
ax.set_xlabel("dₖ  (attention head dimension)")
ax.set_ylabel("mean peak softmax probability")
ax.set_title("Without √dₖ scaling, softmax saturates to one-hot as dₖ grows")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

---

## Part 5 — Multi-Head Attention

Running attention once gives one "view" of token relationships. **Multi-Head Attention** (MHA) runs $H$ parallel attention heads, each with its own $W_Q^{(h)}, W_K^{(h)}, W_V^{(h)}$ projections. The outputs are concatenated and linearly projected:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W_O$$

**Why multiple heads?** Each head can specialise on a different relationship type:

- Head 0 might track syntactic dependencies (subject → verb)
- Head 1 might track coreference ("the cat" … "it")
- Head 2 might capture positional proximity

With $H = 2$ heads on a $d = D_{WORK} = 16$ model, each head works in $d_k = 8$ dimensions.


In [ ]:
# ── Working model constants ───────────────────────────────────────────────────
D_WORK = 16  # functional model dimension
NUM_HEADS = 2  # attention heads
D_HEAD = D_WORK // NUM_HEADS  # 8 per head
D_FF = 32  # feed-forward hidden size


class MultiHeadAttention(keras.layers.Layer):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = layers.Dense(d_model, use_bias=False)
        self.W_K = layers.Dense(d_model, use_bias=False)
        self.W_V = layers.Dense(d_model, use_bias=False)
        self.W_O = layers.Dense(d_model, use_bias=False)

    def split_heads(self, x):
        # (batch, seq, d_model) → (batch, heads, seq, d_head)
        B, S = tf.shape(x)[0], tf.shape(x)[1]
        x = tf.reshape(x, (B, S, self.n_heads, self.d_head))
        return tf.transpose(x, [0, 2, 1, 3])

    def call(self, x, mask=None):
        B, S = tf.shape(x)[0], tf.shape(x)[1]
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))

        scores = tf.matmul(Q, K, transpose_b=True) / math.sqrt(self.d_head)
        if mask is not None:
            scores = tf.where(
                mask[tf.newaxis, tf.newaxis, :, :],
                tf.constant(float("-inf"), scores.dtype),
                scores,
            )
        attn_w = tf.nn.softmax(scores, axis=-1)

        out = tf.matmul(attn_w, V)  # (B, heads, seq, d_head)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, (B, S, self.d_model))
        return self.W_O(out), attn_w


# ── Demo ─────────────────────────────────────────────────────────────────────
tf.random.set_seed(42)
mha = MultiHeadAttention(D_WORK, NUM_HEADS)

# Project our 3D embeddings into D_WORK via a simple linear pad
proj = layers.Dense(D_WORK, use_bias=False)
x_work = proj(embs)[tf.newaxis, ...]  # (1, 6, 16)

mha_out, head_weights = mha(x_work)
print(
    f"MHA output shape: {tuple(mha_out.shape)}   "
    f"head_weights shape: {tuple(head_weights.shape)}"
)

# Compare the two heads' attention patterns
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].numpy()  # (seq, seq)
    sns.heatmap(
        w_h,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="Purples",
        xticklabels=TOKENS,
        yticklabels=TOKENS,
        linewidths=0.5,
        cbar=False,
    )
    ax.set_title(f"Head {h} attention weights")
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle(
    "Multi-Head Attention — each head learns a different relationship",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

#### 🔮 Predict first — can one head do two jobs?

A single attention head produces **one** score matrix → **one** softmax pattern. Suppose you want every token to attend to **both** its previous token (a _positional_ relation) **and** its most semantically similar token (a _content_ relation).

**Predict:** can one head satisfy both relations at once, or must it pick one? Run the next cell — we build two hand-crafted heads and _measure_ whether either can do the other's job.


In [ ]:
# ── PROVING the multi-head claim — step 1: two heads, two patterns ───────────
#
# Claim under test: "each head can specialise on a DIFFERENT relationship."
# We hand-build two heads embodying two distinct relations, then show their
# patterns genuinely differ. (Next cell tests whether one head could do both.)
# In a trained model the loss discovers these specialisations by itself — here
# we construct them so the claim is testable rather than hand-waved.

tf.random.set_seed(0)
S = SEQ_LEN
X = embs  # (S, 3) our token vectors
V_shared = tf.matmul(X, tf.random.normal((3, 3)))  # the values every head reads from

# --- Relation P (positional): attend to the PREVIOUS token -------------------
prev_idx = np.array([max(i - 1, 0) for i in range(S)])
scores_pos = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_pos[i, prev_idx[i]] = 9.0
A_pos = tf.nn.softmax(tf.constant(scores_pos), axis=-1)

# --- Relation C (content): attend to the most SEMANTICALLY SIMILAR token -----
sim = tf.matmul(X, X, transpose_b=True).numpy()
np.fill_diagonal(sim, -1e9)  # never match yourself
near_idx = sim.argmax(-1)
scores_con = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S):
    scores_con[i, near_idx[i]] = 9.0
A_con = tf.nn.softmax(tf.constant(scores_con), axis=-1)

# Are the two patterns genuinely different?
corr = np.corrcoef(A_pos.numpy().flatten(), A_con.numpy().flatten())[0, 1]
print(
    f"Correlation between the two head patterns: {corr:+.3f}   (≈0 ⇒ different information)"
)

# --- Visualise the two specialised patterns ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, A, title, cmap in [
    (
        axes[0],
        A_pos.numpy(),
        "Head P — positional (attend to previous token)",
        "Greens",
    ),
    (
        axes[1],
        A_con.numpy(),
        "Head C — content (attend to most similar token)",
        "Purples",
    ),
]:
    sns.heatmap(
        A,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        xticklabels=TOKENS,
        yticklabels=TOKENS,
        linewidths=0.5,
        cbar=False,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Two heads, two DIFFERENT relations", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

#### So they differ — but could a _single_ head carry both?

Each head yields exactly **one** output per token. If one head tried to serve both relations, it would have to reconstruct both targets at once. Let's measure how well each head recovers each relation.


In [ ]:
# ── PROVING the multi-head claim — step 2: one head can't do both ────────────
# Each head produces ONE output. Can it match BOTH relation targets at once?
target_prev = tf.gather(V_shared, prev_idx)  # what relation P should deliver
target_near = tf.gather(V_shared, near_idx)  # what relation C should deliver
out_pos = tf.matmul(A_pos, V_shared)
out_con = tf.matmul(A_con, V_shared)


def mse(a, b):
    return float(tf.reduce_mean((a - b) ** 2))


print("Reconstruction error (lower = that relation is captured):")
print(
    f"  Head P alone → previous-token target : {mse(out_pos, target_prev):.4f}   ✓ nails it"
)
print(
    f"  Head P alone → similar-token  target : {mse(out_pos, target_near):.4f}   ✗ misses it"
)
print(
    f"  Head C alone → previous-token target : {mse(out_con, target_prev):.4f}   ✗ misses it"
)
print(
    f"  Head C alone → similar-token  target : {mse(out_con, target_near):.4f}   ✓ nails it"
)
print()
print("  → A single head serves ONE relation well, never both.")
print("  → Concatenating [Head P ; Head C] delivers BOTH targets in parallel.")
print("  → That is why H heads exist: H independent relations, computed at once.")

### 🧪 Your turn — heads

You proved one head can't do two jobs. Now feel it: dial the number of heads up and down and watch how independent their patterns become.


In [ ]:
# 🧪 EXERCISE 2 — how many heads?
# 👉 CHANGE `n_heads` (must divide D_WORK = 16: try 1, 2, 4, 8) and PREDICT:
#    more heads = more independent relations captured at once.
n_heads = 1  # ← try 1, then 4, then 8

tf.random.set_seed(42)
mha_ex = MultiHeadAttention(D_WORK, n_heads)
_, w_ex = mha_ex(x_work)  # (1, n_heads, seq, seq)
print(
    f"{n_heads} head(s) → {n_heads} attention pattern(s), each {D_WORK // n_heads}-dim wide."
)

pats = tf.reshape(w_ex[0], (n_heads, -1)).numpy()
if n_heads > 1:
    cc = np.corrcoef(pats)
    off = cc[np.triu_indices(n_heads, 1)].mean()
    print(
        f"avg correlation between heads: {off:+.2f}   (near 0 ⇒ each head does its own job)"
    )
else:
    print(
        "with 1 head there is only ONE relation to go around — no division of labour."
    )

---

## Part 6 — Feed-Forward Network & Layer Normalisation

After MHA the output passes through two more components:

### Feed-Forward Network (FFN)

A position-wise (applied independently to each token) 2-layer MLP with a ReLU/GELU:

$$\text{FFN}(x) = \max(0,\, x W_1 + b_1)\, W_2 + b_2$$

Expands by 4× ($D_{WORK} \to D_{FF}$) then projects back. This "up-project then down-project" pattern adds non-linear transformation capacity.

### Layer Normalisation

Applied **before** each sub-layer in modern pre-norm transformers (GPT-2 and later):

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where $\mu$ and $\sigma^2$ are the mean and variance taken **across the feature dimension** of a single token (the $\epsilon$ lives _inside_ the square root, guarding the variance — not added to the standard deviation). Normalises each token's representation independently (not over the batch). Keeps activations well-scaled as depth increases — critical for training stability.

### Residual Connections

Both MHA and FFN are wrapped in **residual (skip) connections**:

$$y = \text{LayerNorm}(x + \text{SubLayer}(x))$$

The identity path $x$ guarantees gradient flow through deep stacks.


In [ ]:
class FeedForward(keras.layers.Layer):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = keras.Sequential(
            [
                layers.Dense(d_ff, activation="gelu"),
                layers.Dense(d_model),
            ]
        )

    def call(self, x):
        return self.net(x)


# ── Visualise LayerNorm effect ────────────────────────────────────────────────
tf.random.set_seed(42)

ffn = FeedForward(D_WORK, D_FF)
norm = layers.LayerNormalization(epsilon=1e-5)

x_raw = x_work[0]  # (6, 16)
x_after = ffn(x_raw)  # (6, 16) raw FFN output
x_normed = norm(x_after)  # (6, 16) after LayerNorm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, title in zip(
    axes,
    [x_raw, x_after, x_normed],
    ["Input to FFN", "FFN output (raw)", "After LayerNorm"],
):
    data_np = data.numpy()
    for j, token in enumerate(TOKENS):
        vals = data_np[j]
        mu = vals.mean()
        sig = vals.std()
        ax.plot(vals, alpha=0.7, label=f"{token}  μ={mu:.2f}, σ={sig:.2f}")
    ax.set_title(title)
    ax.set_xlabel("Hidden dimension")
    ax.set_ylabel("Activation value")
    ax.legend(fontsize=7)
    ax.axhline(0, color="black", lw=0.5, ls="--")

plt.suptitle(
    "FFN activations before and after LayerNorm", fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("LayerNorm centres and normalises each token slice.")
print("Mean and standard deviation across dimensions after LN:")
x_normed_np = x_normed.numpy()
for j, token in enumerate(TOKENS):
    v = x_normed_np[j]
    print(f"  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}")

---

## Part 7 — Full Transformer Block & RNN Comparison

A single **Transformer Block** wires MHA + FFN together with layer norm and residuals:

```
x ──→ LayerNorm ──→ MHA ──→ (+x) ──→ LayerNorm ──→ FFN ──→ (+x) ──→ output
         ↑_____________↗                  ↑___________________________↗
              residual                           residual
```

Stack $L$ of these blocks = the full encoder/decoder stack.

### Why not just use an RNN?

| Feature           | RNN                            | Transformer                         |
| ----------------- | ------------------------------ | ----------------------------------- |
| Parallelism       | ❌ sequential over time        | ✅ all positions at once            |
| Long-range memory | ❌ vanishes through time steps | ✅ direct attention at any distance |
| Training          | ❌ BPTT through long sequences | ✅ simple backprop                  |
| Context window    | Limited by hidden state        | Limited only by memory              |


In [ ]:
class TransformerBlock(keras.layers.Layer):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)
        self.ffn = FeedForward(d_model, d_ff)

    def call(self, x, mask=None):
        # Pre-norm MHA + residual
        mha_out, attn_w = self.mha(self.norm1(x), mask=mask)
        x = x + mha_out
        # Pre-norm FFN + residual
        x = x + self.ffn(self.norm2(x))
        return x, attn_w


tf.random.set_seed(42)
block1 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)
block2 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)

x0 = tf.identity(x_work)  # (1, 6, 16)
x1, aw1 = block1(x0)
x2, aw2 = block2(x1)

print(f"Input  → Block 1 → Block 2:")
print(f"  x0: {tuple(x0.shape)}  norm={float(tf.norm(x0)):.3f}")
print(f"  x1: {tuple(x1.shape)}  norm={float(tf.norm(x1)):.3f}")
print(f"  x2: {tuple(x2.shape)}  norm={float(tf.norm(x2)):.3f}")

# Visualise how representation norms evolve token-by-token
fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    "Layer 0 (input)": tf.norm(x0[0], axis=-1).numpy(),
    "Layer 1 output": tf.norm(x1[0], axis=-1).numpy(),
    "Layer 2 output": tf.norm(x2[0], axis=-1).numpy(),
}
x_pos = np.arange(SEQ_LEN)
width = 0.25
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k * width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Representation L2 norm")
ax.set_title("How token representations grow through transformer blocks")
ax.legend()
plt.tight_layout()
plt.show()

#### 🔮 Predict first — does the skip connection really matter?

We'll stack **24** simple layers and read the gradient that reaches **layer 1** (the one furthest from the loss), with and without the skip connection $x + \text{SubLayer}(x)$.

**Predict:** without the skip, will the gradient at layer 1 be roughly the same as with it, a little smaller, or vanishingly small? Then run the cell and read the log-scale plot.


In [ ]:
# ── DEMONSTRATING why residual connections make depth trainable ──────────────
#
# Claim under test: "the identity path guarantees gradient flow through deep stacks."
# We build the SAME 24-layer stack twice — once with the skip  x + f(x)  and once
# without it (plain f(x)) — then read the gradient norm that reaches each layer
# from a single loss at the top. This is the classic vanishing-gradient picture.

DEPTH = 24
d = D_WORK


class ProbeBlock(keras.layers.Layer):
    def __init__(self, d, residual):
        super().__init__()
        self.lin = layers.Dense(d)
        self.residual = residual

    def call(self, x):
        y = tf.tanh(self.lin(x))
        return x + y if self.residual else y


def gradient_reaching_each_layer(residual):
    tf.random.set_seed(0)  # identical init for a fair race
    blocks = [ProbeBlock(d, residual) for _ in range(DEPTH)]
    x = tf.random.normal((1, d))
    with tf.GradientTape() as tape:
        h = x
        for b in blocks:
            h = b(h)
        loss = tf.reduce_mean(h**2)  # a scalar at the very top
    # ‖∂loss/∂W‖ for each block = how much learning signal survived to that depth
    grads = tape.gradient(loss, [b.lin.kernel for b in blocks])
    return [float(tf.norm(g)) for g in grads]


g_res = gradient_reaching_each_layer(residual=True)
g_plain = gradient_reaching_each_layer(residual=False)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(
    range(1, DEPTH + 1), g_plain, "o-", color="tomato", lw=2, label="WITHOUT residual"
)
ax.plot(range(1, DEPTH + 1), g_res, "o-", color="seagreen", lw=2, label="WITH residual")
ax.set_yscale("log")
ax.set_xlabel("Layer (1 = furthest from the loss, i.e. closest to the input)")
ax.set_ylabel("‖gradient‖ reaching this layer  (log scale)")
ax.set_title("Residual connections keep gradients alive all the way to layer 1")
ax.legend()
plt.tight_layout()
plt.show()

print("Gradient norm reaching layer 1 (the earliest, hardest-to-train layer):")
print(f"  WITHOUT residual: {g_plain[0]:.2e}   ← vanished")
print(f"  WITH    residual: {g_res[0]:.2e}   ← healthy")
print(f"  The skip path delivers ~{g_res[0]/max(g_plain[0], 1e-30):.1e}× more gradient")
print("  to the earliest layer. Without it, deep stacks simply cannot learn their")
print("  bottom layers — which is why every modern transformer wraps each sublayer")
print("  in  x + SubLayer(x).")

### 🧪 Your turn — depth

The skip connection's value grows with depth. Push the stack deeper and watch the no-residual gradient collapse while the residual one stays alive.


In [ ]:
# 🧪 EXERCISE 3 — how deep can you go WITHOUT residuals?
# 👉 CHANGE `DEPTH` (try 8, 24, 60) and PREDICT how far the gradient survives.
DEPTH = 40  # ← the deeper you go, the more the skip path matters

g_res_ex = gradient_reaching_each_layer(residual=True)
g_plain_ex = gradient_reaching_each_layer(residual=False)
print(f"At depth {DEPTH}, gradient reaching layer 1 (the hardest to train):")
print(f"  WITHOUT residual: {g_plain_ex[0]:.2e}")
print(f"  WITH    residual: {g_res_ex[0]:.2e}")
print(
    f"  ratio: {g_res_ex[0] / max(g_plain_ex[0], 1e-30):.1e}×   "
    f"→ without the skip, deep stacks starve their bottom layers."
)

---

## Part 8 — Mini Language Model: Training & Inference

We now wire everything together into a **Mini Language Model** — a decoder-only transformer that learns to predict the next token. This is the architecture of GPT, LLaMA, Mistral etc.

**Training task**: given a context window, predict the next token.  
For our sentence `"the cat sat on the mat"` we create pairs like:

- `["the"]` → `"cat"`
- `["the", "cat"]` → `"sat"`
- etc.

The model outputs a probability distribution over the vocabulary at each position; we minimise cross-entropy loss.


In [ ]:
class MiniLM(keras.Model):
    """
    Decoder-only transformer language model.
    Architecture: token_emb → sinusoidal_PE → n_layers×TransformerBlock → lm_head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.blocks = [
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ]
        self.norm_out = layers.LayerNormalization(epsilon=1e-5)
        # Sinusoidal PE, kept as a constant (not a trainable weight)
        self.pe = sinusoidal_pe(max_seq, d_model)

    def call(self, token_ids, return_attn=False):
        """
        token_ids: (batch, seq_len)  int tensor
        Returns logits: (batch, seq_len, vocab_size)
        """
        S = int(token_ids.shape[1])
        x = self.token_emb(token_ids) + self.pe[:S]  # (B, S, d_model)

        # Causal mask — each position sees only previous tokens
        causal_mask = tf.constant(np.triu(np.ones((S, S)), k=1).astype(bool))
        all_attn = []
        for block in self.blocks:
            x, aw = block(x, mask=causal_mask)
            all_attn.append(aw)

        x = self.norm_out(x)
        # Weight tying: reuse the token embedding matrix as the output projection
        logits = tf.matmul(x, self.token_emb.embeddings, transpose_b=True)

        if return_attn:
            return logits, all_attn
        return logits


tf.random.set_seed(42)
model = MiniLM(
    vocab_size=VOCAB_SIZE, d_model=D_WORK, n_heads=NUM_HEADS, d_ff=D_FF, n_layers=2
)
_ = model(tf.constant([TOKEN_IDS]))  # build the model (creates weights)
n_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"MiniLM — {n_params} trainable parameters")
model.summary()

In [ ]:
# ── Training data — (context, next_token) pairs from our sentence ─────────────
full_corpus = [
    "the cat sat on the mat",
    "the dog ran over the fence",
    "a big cat jumped over the fence",
    "a dog sat on the mat",
    "the cat jumped over the fence",
    "the big dog ran on the mat",
]

TRAIN_PAIRS = []
for sentence in full_corpus:
    ids = encode(sentence)
    for end in range(1, len(ids)):
        context = ids[:end]
        target = ids[end]
        TRAIN_PAIRS.append((context, target))

print(f"Training pairs: {len(TRAIN_PAIRS)}")
for ctx, tgt in TRAIN_PAIRS[:5]:
    print(f'  {[IDX2WORD[i] for i in ctx]} → "{IDX2WORD[tgt]}"')
print("  ...")

Now the training loop: full-batch gradient descent, predicting each next token from the position before it.


In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
def pad_collate(pairs, pad_id=0):
    max_len = max(len(ctx) for ctx, _ in pairs)
    xs, ys = [], []
    for ctx, tgt in pairs:
        pad = [pad_id] * (max_len - len(ctx))
        xs.append(pad + ctx)
        ys.append(tgt)
    return tf.constant(xs, dtype=tf.int32), tf.constant(ys, dtype=tf.int32)


tf.random.set_seed(42)
model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
optim = keras.optimizers.Adam(learning_rate=3e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

EPOCHS = 300
BATCH_SIZE = len(TRAIN_PAIRS)  # full-batch for this tiny dataset
loss_history = []
acc_history = []

x, y = pad_collate(TRAIN_PAIRS)  # (N, max_ctx_len), (N,)
for epoch in range(EPOCHS):
    with tf.GradientTape() as tape:
        logits = model(x)  # (N, max_ctx_len, vocab_size)
        last_logits = logits[:, -1, :]  # predict NEXT token from last position
        loss = loss_fn(y, last_logits)

    grads = tape.gradient(loss, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optim.apply_gradients(zip(grads, model.trainable_variables))

    if (epoch + 1) % 10 == 0:
        preds = tf.argmax(last_logits, axis=-1, output_type=tf.int32)
        acc = float(tf.reduce_mean(tf.cast(preds == y, tf.float32)))
        loss_history.append(float(loss))
        acc_history.append(acc)
        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1:4d} | loss={float(loss):.4f} | acc={acc:.2%}")

print("\nTraining complete.")

In [ ]:
# ── Training loss & accuracy plot ─────────────────────────────────────────────
epochs_logged = list(range(10, EPOCHS + 1, 10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(epochs_logged, loss_history, color="royalblue", lw=2)
ax.fill_between(epochs_logged, loss_history, alpha=0.15, color="royalblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss")

ax2 = axes[1]
ax2.plot(epochs_logged, [a * 100 for a in acc_history], color="mediumseagreen", lw=2)
ax2.fill_between(
    epochs_logged, [a * 100 for a in acc_history], alpha=0.15, color="mediumseagreen"
)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Next-Token Prediction Accuracy")
ax2.set_ylim(0, 105)
ax2.axhline(100, color="grey", ls="--", lw=0.8)

plt.suptitle("MiniLM Training Progress", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---

## Part 9 — Inference: Autoregressive Token Generation

At inference time a language model generates text **one token at a time**:

1. Feed the current context (all tokens generated so far) into the model
2. Take the logits at the **last position** — this is the model's prediction for the next token
3. Apply temperature scaling + softmax to get a probability distribution
4. Sample (or take argmax = greedy) from that distribution
5. Append the new token to the context → go to step 1

This is called **autoregressive** generation. Each new token becomes part of the context for the next step.

**Temperature** controls randomness:

- $T \to 0$: always pick the highest-probability token (deterministic / greedy)
- $T = 1.0$: sample from the raw model distribution
- $T > 1.0$: flatten the distribution → more random / creative


In [ ]:
# Inference mode — Keras layers use training=False by default when called directly.


def generate_next(context_words, temperature=1.0):
    """Single next-token generation step with probability bar chart."""
    ids = tf.constant([encode(" ".join(context_words))], dtype=tf.int32)
    logits = model(ids)  # (1, seq, vocab)
    last = logits[0, -1, :]  # (vocab,)
    probs = tf.nn.softmax(last / max(temperature, 1e-6), axis=-1).numpy()

    # Bar chart of top-k candidates
    topk_idx = probs.argsort()[::-1][:8]
    top_words = [IDX2WORD[int(i)] for i in topk_idx]
    top_probs = probs[topk_idx]

    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.barh(
        top_words[::-1],
        top_probs[::-1],
        color=["gold" if w == top_words[0] else "steelblue" for w in top_words[::-1]],
    )
    ax.set_xlabel("Probability")
    ax.set_title(
        f"Next token probabilities | context: {context_words}  T={temperature}"
    )
    ax.set_xlim(0, 1.0)
    for bar, prob in zip(bars, top_probs[::-1]):
        ax.text(
            bar.get_width() + 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{prob:.3f}",
            va="center",
            fontsize=9,
        )
    plt.tight_layout()
    plt.show()

    best = IDX2WORD[int(probs.argmax())]
    print(f'  Greedy prediction: "{best}"')
    return best


# ── Demo: step-by-step generation ────────────────────────────────────────────
print("=== Autoregressive generation ===")
print()
context = ["the"]
for step in range(5):
    print(f"Step {step+1}: context = {context}")
    next_tok = generate_next(context, temperature=0.8)
    context.append(next_tok)
    print()

print(f'Generated sequence: {" ".join(context)}')

### From toy to real — same mechanism, bigger numbers

Everything you've built used tiny dimensions so the vectors stayed readable. A production model is the **identical machinery** scaled up — nothing new is added, the numbers just get bigger. Here's the exact mapping before we load a real GPT-2.


In [ ]:
# Toy (this notebook) vs. a real model (GPT-2 / DistilGPT-2) — the SAME components
rows = [
    ("embedding dim  d_model", D_MODEL, D_WORK, 768),
    ("attention heads", "—", NUM_HEADS, 12),
    ("dim per head  d_head", "—", D_HEAD, 64),
    ("feed-forward hidden", "—", D_FF, 3072),
    ("transformer layers", "—", 2, 12),
    ("vocabulary size", VOCAB_SIZE, VOCAB_SIZE, 50257),
]
print(f'{"component":<24}{"viz":>8}{"toy model":>12}{"GPT-2":>10}')
print("  " + "─" * 52)
for name, viz, toy, real in rows:
    print(f"{name:<24}{str(viz):>8}{str(toy):>12}{str(real):>10}")
print()
print(
    "Same Q/K/V, same softmax(QKᵀ/√dₖ), same multi-head split, same residual + LayerNorm,"
)
print("same FFN — just wider vectors and more layers. If you understood the toy, you")
print("understand GPT-2. The next section runs the real thing.")

---

## Part 10 — Cracking Open distilgpt2

Now we scale from our toy model to a real one: **DistilGPT-2** — a distilled version of GPT-2 with:

- 6 transformer blocks
- 12 attention heads per block
- d_model = 768
- ~82M parameters

We will:

1. Load the model and inspect its architecture
2. Run a forward pass with `output_attentions=True` to capture all 6 layers × 12 heads
3. Visualise the attention patterns
4. Plot the next-token probability distribution


In [ ]:
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

gpt_tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
gpt2 = TFGPT2LMHeadModel.from_pretrained("distilgpt2")

# Architecture overview
cfg = gpt2.config
print("=== DistilGPT-2 Architecture ===")
print(f"  n_layer       : {cfg.n_layer}")
print(f"  n_head        : {cfg.n_head}")
print(f"  n_embd (d_model): {cfg.n_embd}")
print(f"  vocab_size    : {cfg.vocab_size}")
print(f"  n_positions   : {cfg.n_positions}  (max context)")
print()
n_params = gpt2.num_parameters()
print(f"  Total parameters: {n_params:,}  (~{n_params/1e6:.1f}M)")
print()

# Layer names
print("Top-level layers:")
for layer in gpt2.layers:
    print(f"  {layer.name}: {layer.__class__.__name__}")
print()
print("First transformer block sub-modules:")
block0 = gpt2.transformer.h[0]
for name in ["ln_1", "attn", "ln_2", "mlp"]:
    mod = getattr(block0, name)
    print(f"  h[0].{name}: {mod.__class__.__name__}")

In [ ]:
# ── Run inference with all attentions captured ─────────────────────────────────
PROMPT = "The cat sat on the"
inputs = gpt_tokenizer(PROMPT, return_tensors="tf")
input_ids = inputs["input_ids"]

gpt_tokens = [gpt_tokenizer.decode([int(t)]) for t in input_ids[0]]
print(f"Prompt       : {PROMPT!r}")
print(f"GPT-2 tokens : {gpt_tokens}")
print(f"Token IDs    : {input_ids[0].numpy().tolist()}")
print()

out = gpt2(**inputs, output_attentions=True)

# out.attentions: tuple of 6, each (1, 12, seq_len, seq_len)
print(f"Number of attention layers returned: {len(out.attentions)}")
print(f"Each layer shape: {tuple(out.attentions[0].shape)}")
print(f"  (batch=1, n_heads=12, seq={len(gpt_tokens)}, seq={len(gpt_tokens)})")
print()

# Logits for next-token prediction (last position)
next_logits = out.logits[0, -1, :]
next_probs = tf.nn.softmax(next_logits, axis=-1)
top10 = tf.math.top_k(next_probs, k=10)

print(f'Top-10 next token predictions after "{PROMPT}":')
for rank, (tid, prob) in enumerate(zip(top10.indices.numpy(), top10.values.numpy()), 1):
    tok = gpt_tokenizer.decode([int(tid)])
    print(f"  {rank:2d}. {tok!r:<20} {float(prob):.4f}")

In [ ]:
# ── Attention pattern visualisation — all 6 layers, mean over heads ───────────
n_layers = len(out.attentions)
seq_len = len(gpt_tokens)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for layer_idx, (layer_attn, ax) in enumerate(zip(out.attentions, axes)):
    # Mean over the 12 heads → (seq_len, seq_len)
    mean_attn = tf.reduce_mean(layer_attn[0], axis=0).numpy()
    sns.heatmap(
        mean_attn,
        ax=ax,
        cmap="viridis",
        xticklabels=gpt_tokens,
        yticklabels=gpt_tokens,
        linewidths=0.3,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 8},
    )
    ax.set_title(f"Layer {layer_idx}  (mean over 12 heads)")
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

plt.suptitle(
    f'DistilGPT-2 Attention Maps — all 6 layers\n"{PROMPT}"',
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Head diversity — compare individual heads in layer 0 ──────────────────────
layer0_attn = out.attentions[0][0].numpy()  # (12, seq, seq)

# Show first 4 heads
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for h, ax in enumerate(axes):
    sns.heatmap(
        layer0_attn[h],
        ax=ax,
        cmap="Blues",
        xticklabels=gpt_tokens,
        yticklabels=gpt_tokens,
        linewidths=0.3,
        annot=True,
        fmt=".2f",
        annot_kws={"size": 8},
    )
    ax.set_title(f"Layer 0, Head {h}")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
plt.suptitle(
    "Individual attention heads within Layer 0 — each specialises differently",
    fontsize=11,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print()
print("Notice: some heads attend heavily to the immediately preceding token (local),")
print("others spread attention broadly or focus on specific syntactic relations.")

In [ ]:
# ── Next-token probability bar chart ─────────────────────────────────────────
top_n = 15
topk_gpt = tf.math.top_k(next_probs, k=top_n)
top_tokens_gpt = [gpt_tokenizer.decode([int(t)]) for t in topk_gpt.indices.numpy()]
top_probs_gpt = topk_gpt.values.numpy()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["gold"] + ["steelblue"] * (top_n - 1)
bars = ax.barh(
    top_tokens_gpt[::-1], top_probs_gpt[::-1], color=colors[::-1], alpha=0.85
)
ax.set_xlabel("Probability")
ax.set_title(
    f'DistilGPT-2 next token distribution\nPrompt: "{PROMPT}"',
    fontsize=11,
    fontweight="bold",
)
for bar, prob in zip(bars, top_probs_gpt[::-1]):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f"{prob:.4f}",
        va="center",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print()
print(f"Model picks: {top_tokens_gpt[0]!r}  with p={top_probs_gpt[0]:.4f}")
print()
print("This is exactly what autoregressive generation does at every step:")
print("compute this distribution → sample/argmax → append → repeat.")

In [ ]:
# ── Full autoregressive generation with distilgpt2 ────────────────────────────


def gpt2_generate(
    prompt: str, max_new_tokens: int = 15, temperature: float = 0.8, top_k: int = 50
):
    """Generate tokens one at a time, printing each step."""
    ids = gpt_tokenizer.encode(prompt, return_tensors="tf")
    print(f"Prompt: {prompt!r}")
    print(f"Starting IDs: {ids[0].numpy().tolist()}")
    print()

    generated = ids
    for step in range(max_new_tokens):
        out_step = gpt2(generated)
        logits = out_step.logits[0, -1, :]

        # Temperature + top-k filtering
        logits = logits / max(temperature, 1e-8)
        if top_k > 0:
            top_vals, top_idx = tf.math.top_k(logits, k=top_k)
            filtered = tf.ones_like(logits) * float("-inf")
            filtered = tf.tensor_scatter_nd_update(
                filtered, top_idx[:, tf.newaxis], top_vals
            )
            logits = filtered

        probs = tf.nn.softmax(logits, axis=-1)
        # Sample from the (filtered) logits
        next_id = int(tf.random.categorical(logits[tf.newaxis, :], num_samples=1)[0, 0])
        next_tok = gpt_tokenizer.decode([next_id])

        print(
            f'  Step {step+1:2d} → token ID {next_id:5d}  "{next_tok}"  '
            f"  p={float(probs[next_id]):.4f}"
        )

        generated = tf.concat(
            [generated, tf.constant([[next_id]], dtype=generated.dtype)], axis=1
        )

        if next_id == gpt_tokenizer.eos_token_id:
            print("  [EOS — stopping]")
            break

    final_text = gpt_tokenizer.decode(generated[0].numpy(), skip_special_tokens=True)
    print()
    print(f"Final: {final_text!r}")
    return final_text


gpt2_generate("The cat sat on the", max_new_tokens=12, temperature=0.7)

---

## Summary — The Complete Transformer Journey

We've traced every component from raw words to generated tokens:

| Step | Component                        | What happens                                                        |
| ---- | -------------------------------- | ------------------------------------------------------------------- |
| 1    | **Tokeniser**                    | Text → integer IDs                                                  |
| 2    | **Token Embedding**              | IDs → dense vectors in semantic space                               |
| 3    | **Positional Encoding**          | Add position signal (sin/cos or RoPE rotation)                      |
| 4    | **Q/K/V Projection**             | Three learned views of each token vector                            |
| 5    | **Scaled Dot-Product Attention** | Soft dictionary lookup — compute relevance scores                   |
| 6    | **Multi-Head Attention**         | $H$ parallel attention views concatenated                           |
| 7    | **Feed-Forward Network**         | Per-token nonlinear transformation (up-project + down-project)      |
| 8    | **LayerNorm + Residuals**        | Stabilise activations, guarantee gradient flow                      |
| 9    | **Repeat ×L**                    | Stack $L$ transformer blocks                                        |
| 10   | **LM Head**                      | Project to vocabulary → logits → softmax → probability distribution |
| 11   | **Autoregressive loop**          | Sample next token → append → repeat                                 |

### Key insights to keep

- **Embeddings** are not fixed dictionaries — they are learned, contextualised at each layer
- **RoPE** encodes position by rotating Q and K; only the relative gap survives in dot-products
- **Attention is O(n²)** in sequence length — this is why long-context models are expensive
- **Multi-head attention** lets each head specialise on a different relationship type
- **Residual connections** make depth practical — gradients always have a direct path home
- **Temperature** is the single most intuitive control knob at inference time
